In [2]:
import json
import pandas as pd
import h5py
import numpy as np
import tensorflow as tf
import keras
from keras.layers import Input, Dense, Dropout, LSTM, Flatten, GRU,TimeDistributed, Conv1D, BatchNormalization
from keras.models import Model, Sequential
from keras.optimizers import Adam, SGD
from sklearn.model_selection import train_test_split
import os
import h5py
import matplotlib.pyplot as plt
from keras import regularizers
from tensorflow.keras.regularizers import l1
import ast
from tqdm import tqdm
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform, truncnorm, randint
from sklearn.metrics import make_scorer
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.model_selection import RepeatedStratifiedKFold
from scipy.stats import loguniform
from pandas import read_csv
# from keras.wrappers.scikit_learn import KerasClassifier
from scikeras.wrappers import KerasClassifier
from sklearn.metrics import roc_auc_score
from qkeras import *
import qkeras
from tensorflow.keras.models import load_model

import matplotlib.pyplot as plt

2025-11-11 19:36:05.865427: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-11 19:36:05.867848: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-11 19:36:05.895076: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-11 19:36:05.895116: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-11 19:36:05.896289: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [3]:
# generate test data
X_test = np.load('data/X_test.npy', allow_pickle=True)
y_test = np.load('data/y_test.npy', allow_pickle=True)

X_testzero = np.zeros((len(X_test), 100, 3))

for x in range(len(X_test)):
    for y in range(100):
        for z in range(3):
            if y >= len(X_test[x]):
                break
            X_testzero[x][y][z] = X_test[x][y][z]

# encode labels for 5 classes
num_classes = 5
y_labhot = np.zeros((len(y_test),5))

y_labhot.shape

num = 0
for x in y_test:
  if x == 0:
    y_labhot[num][0] = 1
  elif x == 1: 
    y_labhot[num][1] = 1
  elif x == 2: 
    y_labhot[num][2] = 1
  elif x == 3: 
    y_labhot[num][3] = 1
  elif x == 4: 
    y_labhot[num][4] = 1
  num = num + 1

In [139]:
# create hls4ml model - edited (name)

import hls4ml
import plotting

model = load_model('./gru/Quickdraw5Class1_20.h5')
y_keras = model.predict(X_testzero, batch_size=512)

# fixed<x,y> where x is total number, y is integer
config_name = hls4ml.utils.config_from_keras_model(model, granularity='name', default_precision='ap_fixed<21,10>')

for layer in config_name['LayerName'].keys():
    config_name['LayerName'][layer]['Trace'] = True

    # config_name['LayerName']['gru']['result_t'] = 'ap_fixed<32,4>'

    # config_name['LayerName']['dense_1']['result_t'] = 'ap_fixed<32,8>'

    # config_name['LayerName']['rnn_densef']['result_t'] = 'ap_fixed<32,8>'

    # config_name['LayerName']['softmax']['result_t'] = 'ap_fixed<32,4>'
    # config_name['LayerName']['softmax']['table_t'] = 'ap_fixed<32,4>'
    # config_name['LayerName']['softmax']['exp_table_t'] = 'ap_fixed<32,4>'
    # config_name['LayerName']['softmax']['inv_table_t'] = 'ap_fixed<32,4>'
    # config_name['LayerName']['softmax']['table_size'] = 4096
    # implementation: list [latency,stable,argmax,legacy] (Default: stable)
    config_name['LayerName']['softmax']['implementation'] = 'argmax'

print("-----------------------------------")
print("Configuration")
plotting.print_dict(config_name)
print("-----------------------------------")
hls_model_name = hls4ml.converters.convert_from_keras_model(
    model, hls_config=config_name, backend='Vivado', output_dir='model_1/hls4ml_gru/name', part='xc7vx690tffg1761-2'
)

# hls4ml.utils.plot_model(hls_model_name, show_shapes=True, show_precision=True, to_file=None)

25/25 [==============================] - 1s 19ms/step
-----------------------------------
Configuration
Model
  Precision
    default:         ap_fixed<21,10>
  ReuseFactor:       1
  Strategy:          Latency
  BramFactor:        1000000000
  TraceOutput:       False
LayerName
  input_1
    Trace:           True
    Precision
      result:        auto
  gru
    Trace:           True
    Precision
      result:        auto
      weight:        auto
      bias:          auto
      recurrent_weight:auto
      recurrent_bias:auto
  flatten
    Trace:           True
    Precision
      result:        auto
  dense
    Trace:           True
    Precision
      result:        auto
      weight:        auto
      bias:          auto
  dense_linear
    Trace:           True
    Precision
      result:        auto
  relu_0
    Trace:           True
    Precision
      result:        auto
  dense_1
    Trace:           True
    Precision
      result:        auto
      weight:        auto
      

In [140]:
hls_model_name.compile()
X_testzero_contig = np.ascontiguousarray(X_testzero)
y_hls_name = hls_model_name.predict(X_testzero_contig)

In [141]:
y_pred = np.argmax(y_hls_name, axis=1)
y_true = np.argmax(y_labhot, axis=1)

print("hls4ml name  Accuracy: {}".format(accuracy_score(np.argmax(y_labhot, axis=1), np.argmax(y_hls_name, axis=1))))

from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_true, y_pred, target_names=['ant', 'bee', 'butterfly', 'mosquito', 'snail']))

hls4ml name  Accuracy: 0.91808
              precision    recall  f1-score   support

         ant       0.89      0.95      0.92      2500
         bee       0.96      0.84      0.89      2500
   butterfly       0.97      0.96      0.96      2500
    mosquito       0.81      0.87      0.84      2500
       snail       0.98      0.97      0.97      2500

    accuracy                           0.92     12500
   macro avg       0.92      0.92      0.92     12500
weighted avg       0.92      0.92      0.92     12500



In [142]:
name_predict, name_trace =  hls_model_name.trace(X_testzero)

Recompiling myproject with tracing


KeyboardInterrupt: 

In [ ]:
print(name_trace['rnn_densef'])
print(name_trace['softmax'])

[[ 3.67578125 -1.09765625 -3.2109375   1.953125   -3.1328125 ]
 [ 5.5234375   1.07421875 -3.12109375  1.47265625 -2.8984375 ]
 [ 3.3359375  -0.07421875 -2.51171875  1.60546875 -2.4609375 ]
 ...
 [ 5.96875     3.89453125  3.19140625  5.4140625   9.44921875]
 [ 2.45703125  2.890625    1.5546875   4.4375      5.859375  ]
 [ 2.015625    2.4453125   1.08984375  4.0390625   4.9453125 ]]
[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]


In [ ]:
print(y_hls_name)
print(y_labhot)
print(y_pred)
print(y_true)

print(y_hls_name[0])
print(np.argmax(y_hls_name[0]))

[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]
[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]
[0 0 0 ... 4 4 4]
[0 0 0 ... 4 4 4]
[1. 0. 0. 0. 0.]
0


In [ ]:
from keras.models import Model

layer_name = 'gru'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras gru output:")
print(intermediate_output[10:])

print("hls4ml gru output:")
print(name_trace['gru'][10:])

print("highest value:")
print(np.max(name_trace['gru']))

391/391 [==============================] - 2s 5ms/step
keras gru output:
[[[ 2.04587445e-01 -1.00459494e-01 -1.43263312e-02 ...  3.87110822e-02
    3.81681174e-02 -1.10200591e-01]
  [-1.53629482e-03 -5.67562245e-02 -3.55248153e-02 ...  9.46441442e-02
    1.08454190e-01 -2.47629523e-01]
  [-1.60975933e-01  6.01930209e-02 -6.50504902e-02 ...  8.99203271e-02
    1.30175814e-01 -2.33641073e-01]
  ...
  [ 3.06461215e-01 -1.40483037e-03 -4.28204983e-02 ...  6.08870164e-02
    9.98985767e-02 -2.99903806e-02]
  [ 3.06295156e-01 -1.42287463e-03 -4.26650494e-02 ...  6.06960878e-02
    9.99135971e-02 -2.99272239e-02]
  [ 3.06140840e-01 -1.43661583e-03 -4.25221547e-02 ...  6.05253056e-02
    9.99301821e-02 -2.98723131e-02]]

 [[-2.53054649e-02 -2.76440959e-02 -3.47348079e-02 ...  9.84782279e-02
   -7.95906931e-02 -1.87224984e-01]
  [-2.55999207e-01  8.33048970e-02 -7.43145570e-02 ...  1.19783811e-01
   -8.91697407e-03 -2.67627031e-01]
  [-2.41908148e-01  2.05795720e-01 -1.20704353e-01 ...  1.06482

In [ ]:
model.summary()

Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100, 3)]          0         
                                                                 
 gru (GRU)                   (None, 100, 32)           3552      
                                                                 
 flatten (Flatten)           (None, 3200)              0         
                                                                 
 dense (Dense)               (None, 32)                102432    
                                                                 
 relu_0 (Activation)         (None, 32)                0         
                                                                 
 dropout (Dropout)           (None, 32)                0         
                                                                 
 dense_1 (Dense)             (None, 16)                528   

In [ ]:
print(name_trace.keys())

dict_keys(['gru', 'dense', 'relu_0', 'dense_1', 'relu_1', 'rnn_densef', 'softmax'])


In [ ]:
layer_name = 'dense'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras dense output:")
print(intermediate_output[10:])

print("hls4ml dense output:")
print(name_trace['dense'][10:])

print("highest value:")
print(np.max(name_trace['dense']))

391/391 [==============================] - 2s 5ms/step
keras dense output:
[[ 14.318244  -12.082122    7.710285  ...  15.697724    8.445192
  -11.438699 ]
 [ 13.5980835 -14.581085    5.439097  ...  10.082352   11.007762
   -9.260301 ]
 [  9.8356695   5.347862  -12.592067  ...   3.0161352  13.35141
    3.0368495]
 ...
 [ 19.878902   31.961655    0.9881185 ... -12.682116   37.080128
   15.4717865]
 [ -5.3416953  11.246378    4.8443856 ... -24.01758     8.539798
    8.643997 ]
 [ -3.7657623  12.948891   -1.9831028 ... -13.017563    7.0883365
   11.030282 ]]
hls4ml dense output:
[[ 12.82421875 -14.4609375    2.296875   ...  12.9140625    8.5234375
  -15.34765625]
 [ 10.43359375 -15.76953125  -0.23046875 ...   5.421875     9.92578125
  -11.625     ]
 [  9.25         2.24609375 -16.7109375  ...   2.30859375  13.046875
   -5.87890625]
 ...
 [ 16.2265625   19.28515625  -4.640625   ... -13.22265625  29.60546875
    5.7734375 ]
 [ -5.796875     9.53125     -0.28515625 ... -26.41796875  10.207031

In [ ]:
layer_name = 'relu_0'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras relu_0 output:")
print(intermediate_output[10:])

print("hls4ml relu_0 output:")
print(name_trace['relu_0'][10:])

print("highest value:")
print(np.max(name_trace['relu_0']))

391/391 [==============================] - 2s 5ms/step
keras relu_0 output:
[[14.318244   0.         7.710285  ... 15.697724   8.445192   0.       ]
 [13.5980835  0.         5.439097  ... 10.082352  11.007762   0.       ]
 [ 9.8356695  5.347862   0.        ...  3.0161352 13.35141    3.0368495]
 ...
 [19.878902  31.961655   0.9881185 ...  0.        37.080128  15.4717865]
 [ 0.        11.246378   4.8443856 ...  0.         8.539798   8.643997 ]
 [ 0.        12.948891   0.        ...  0.         7.0883365 11.030282 ]]
hls4ml relu_0 output:
[[12.82421875  0.          2.296875   ... 12.9140625   8.5234375
   0.        ]
 [10.43359375  0.          0.         ...  5.421875    9.92578125
   0.        ]
 [ 9.25        2.24609375  0.         ...  2.30859375 13.046875
   0.        ]
 ...
 [16.2265625  19.28515625  0.         ...  0.         29.60546875
   5.7734375 ]
 [ 0.          9.53125     0.         ...  0.         10.20703125
   1.        ]
 [ 0.         12.05859375  0.         ...  0.      

In [ ]:
layer_name = 'dense_1'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras dense_1 output:")
print(intermediate_output[10:])

print("hls4ml dense_1 output:")
print(name_trace['dense_1'][10:])

print("highest value:")
print(np.max(name_trace['dense_1']))

391/391 [==============================] - 2s 5ms/step
keras dense_1 output:
[[-1.0532380e+01 -2.1051962e+01 -1.9040052e+00 ... -1.1841638e+02
   6.0761776e+00 -3.5079043e+00]
 [-7.7428751e+00 -1.7109451e+01 -1.4814194e+00 ... -1.0375508e+02
   5.0859156e+00 -2.6003139e+00]
 [ 1.6946721e-01  8.6796169e+00 -5.9805446e+00 ... -9.1229263e+01
  -3.6664917e+01 -1.4487172e+01]
 ...
 [-4.8378754e+01  1.5231032e+01 -1.2951339e+01 ... -2.5181790e+02
  -4.3348255e+01 -6.0890831e+01]
 [-3.3554996e+01  5.9839878e+00 -2.2537689e+00 ... -6.4144043e+01
  -6.9774561e+00 -3.1531897e+01]
 [-1.7944662e+01  8.8885489e+00 -7.2256994e+00 ... -5.2191307e+01
  -2.3404613e+01 -3.4113823e+01]]
hls4ml dense_1 output:
[[-4.37500000e+00 -1.52500000e+01  9.76562500e-02 ... -1.05285156e+02
   4.24609375e+00 -1.55859375e+00]
 [-2.44531250e+00 -9.80468750e+00  5.27343750e-01 ... -7.56445312e+01
   3.13281250e+00 -6.79687500e-01]
 [-4.41406250e-01  3.91406250e+00 -2.06640625e+00 ... -8.02187500e+01
  -1.66640625e+01 -4

In [ ]:
layer_name = 'relu_1'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras relu_1 output:")
print(intermediate_output[10:])

print("hls4ml relu_1 output:")
print(name_trace['relu_1'][10:])

print("highest value:")
print(np.max(name_trace['relu_1']))

391/391 [==============================] - 2s 5ms/step
keras relu_1 output:
[[ 0.          0.          0.         ...  0.          6.0761776
   0.        ]
 [ 0.          0.          0.         ...  0.          5.0859156
   0.        ]
 [ 0.16946721  8.679617    0.         ...  0.          0.
   0.        ]
 ...
 [ 0.         15.231032    0.         ...  0.          0.
   0.        ]
 [ 0.          5.983988    0.         ...  0.          0.
   0.        ]
 [ 0.          8.888549    0.         ...  0.          0.
   0.        ]]
hls4ml relu_1 output:
[[0.         0.         0.09765625 ... 0.         4.24609375 0.        ]
 [0.         0.         0.52734375 ... 0.         3.1328125  0.        ]
 [0.         3.9140625  0.         ... 0.         0.         0.        ]
 ...
 [0.         8.16015625 0.         ... 0.         0.         0.        ]
 [0.         5.43359375 0.         ... 0.         0.         0.        ]
 [0.         3.9453125  0.         ... 0.         0.         0.        ]]


In [ ]:
layer_name = 'rnn_densef'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras rnn_densef output:")
print(intermediate_output[10:])

print("hls4ml rnn_densef output:")
print(name_trace['rnn_densef'][10:])

print("highest value:")
print(np.max(name_trace['rnn_densef']))

391/391 [==============================] - 2s 5ms/step
keras rnn_densef output:
[[ 4.194874   -1.2755746  -4.1524005   2.153224   -4.0339003 ]
 [ 3.8533535  -1.0665646  -3.8236024   1.9492545  -3.6614661 ]
 [ 2.485917    2.2474754   0.48152745  1.7406696   3.2173986 ]
 ...
 [ 8.318522    6.4038067   6.1114078   8.701138   15.448932  ]
 [ 2.4594052   3.1684      2.5936377   5.5933843   7.3351808 ]
 [ 1.981692    2.8410673   2.964235    4.8878217   6.8884144 ]]
hls4ml rnn_densef output:
[[ 3.4453125  -1.0859375  -3.18359375  1.96484375 -3.1875    ]
 [ 2.8984375  -0.71484375 -2.7109375   2.14453125 -2.5625    ]
 [ 2.1953125   1.69921875 -0.04296875  2.02734375  2.6015625 ]
 ...
 [ 5.96875     3.89453125  3.19140625  5.4140625   9.44921875]
 [ 2.45703125  2.890625    1.5546875   4.4375      5.859375  ]
 [ 2.015625    2.4453125   1.08984375  4.0390625   4.9453125 ]]
highest value:
20.29296875


In [ ]:
layer_name = 'softmax'
intermediate_layer_model = Model(inputs=model.input,
                                 outputs=model.get_layer(layer_name).output)
intermediate_output = intermediate_layer_model.predict(X_testzero)

print("keras softmax output:")
print(intermediate_output[10:])

print("hls4ml softmax output:")
print(name_trace['softmax'][10:])

print("highest value:")
print(np.max(name_trace['softmax']))

391/391 [==============================] - 2s 5ms/step
keras softmax output:
[[8.8142413e-01 3.7102173e-03 2.0893422e-04 1.1442151e-01 2.3521972e-04]
 [8.6410648e-01 6.3077430e-03 4.0041245e-04 1.2871453e-01 4.7089334e-04]
 [2.2344515e-01 1.7604230e-01 3.0107556e-02 1.0605083e-01 4.6435416e-01]
 ...
 [7.9865078e-04 1.1770813e-04 8.7865636e-05 1.1709128e-03 9.9782491e-01]
 [6.3204528e-03 1.2842830e-02 7.2284401e-03 1.4515029e-01 8.2845801e-01]
 [6.2690261e-03 1.4805466e-02 1.6746080e-02 1.1463472e-01 8.4754467e-01]]
hls4ml softmax output:
[[1. 0. 0. 0. 0.]
 [1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 1.]
 ...
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]
 [0. 0. 0. 0. 1.]]
highest value:
1.0


In [ ]:
print(config_name['LayerName'].keys())

dict_keys(['input_1', 'gru', 'flatten', 'dense', 'dense_linear', 'relu_0', 'dense_1', 'dense_1_linear', 'relu_1', 'rnn_densef', 'rnn_densef_linear', 'softmax'])


In [ ]:
# %matplotlib inline
# from hls4ml.model.profiling import numerical

# numerical(model=model, hls_model=hls_model_name, X=X_test[:1000])